# Grouped-Query Attention Practice

**Topic:** MHA and GQA &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `x` `(2, 10, 32)`: batch 2, sequence 10, model dim 32.
- Projections: `Wq (32, 32)` → 8 query heads × head_dim 4, `Wk, Wv (32, 8)` → **2 KV heads** × head_dim 4, and `Wo (32, 32)` output projection.
- Attention here is **not** masked (full bidirectional).

## Your task

With **NumPy only**, compute GQA:

1. `q = x @ Wq` reshaped to 8 heads of dim 4, with `k = x @ Wk` and `v = x @ Wv` reshaped to 2 heads of dim 4.
2. Query heads 0–3 use KV head 0, and query heads 4–7 use KV head 1 (`np.repeat(k, 4, axis=head_axis)` is the tidy way).
3. Per head: softmax(qkᵀ/√4)v. Merge heads back to `(2, 10, 32)`, apply `Wo`, and set `gqa_out` to the result.

## Evaluation

Max absolute difference from the reference must be below 1e-6.


In [ ]:
import numpy as np

rng = np.random.default_rng(431)
B, N, D = 2, 10, 32
N_Q_HEADS, N_KV_HEADS, HEAD_DIM = 8, 2, 4

x = rng.normal(0, 1.0, (B, N, D))
Wq = rng.normal(0, 0.3, (D, N_Q_HEADS * HEAD_DIM))
Wk = rng.normal(0, 0.3, (D, N_KV_HEADS * HEAD_DIM))
Wv = rng.normal(0, 0.3, (D, N_KV_HEADS * HEAD_DIM))
Wo = rng.normal(0, 0.3, (N_Q_HEADS * HEAD_DIM, D))
print(f"x {x.shape}. KV cache per token: {2 * N_KV_HEADS * HEAD_DIM} floats vs {2 * N_Q_HEADS * HEAD_DIM} for full MHA")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to solve the task, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: compare against an independent reference implementation
# =====================================================================
import numpy as np

if "gqa_out" not in dir():
    raise NameError("Define `gqa_out` in the solution cell before running this cell.")

def _softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

_q = (x @ Wq).reshape(B, N, N_Q_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
_k = (x @ Wk).reshape(B, N, N_KV_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
_v = (x @ Wv).reshape(B, N, N_KV_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
_k = np.repeat(_k, N_Q_HEADS // N_KV_HEADS, axis=1)
_v = np.repeat(_v, N_Q_HEADS // N_KV_HEADS, axis=1)
_w = _softmax(_q @ _k.transpose(0, 1, 3, 2) / np.sqrt(HEAD_DIM))
_ref = ((_w @ _v).transpose(0, 2, 1, 3).reshape(B, N, N_Q_HEADS * HEAD_DIM)) @ Wo

_out = np.asarray(gqa_out, dtype=float)
if _out.shape != _ref.shape:
    raise ValueError(f"`gqa_out` has shape {_out.shape}, expected {_ref.shape}.")
_err = float(np.max(np.abs(_out - _ref)))
print("=" * 52)
print(f"Max absolute error vs reference: {_err:.2e}   (need < 1e-6)")
print("=" * 52)
print("PASSED. You just implemented GQA!" if _err < 1e-6 else "NOT PASSED yet. Check the head reshapes and the KV sharing.")


## Solution

Reshape queries into 8 heads and keys/values into 2 heads, then `np.repeat` the KV heads 4x along the head axis so each group of query heads shares one KV head. The rest is vanilla per-head softmax attention plus the output projection.

```python
import numpy as np

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

q = (x @ Wq).reshape(B, N, N_Q_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
k = (x @ Wk).reshape(B, N, N_KV_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
v = (x @ Wv).reshape(B, N, N_KV_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)

# each group of 4 query heads shares one KV head
k = np.repeat(k, N_Q_HEADS // N_KV_HEADS, axis=1)
v = np.repeat(v, N_Q_HEADS // N_KV_HEADS, axis=1)

w = softmax(q @ k.transpose(0, 1, 3, 2) / np.sqrt(HEAD_DIM))
merged = (w @ v).transpose(0, 2, 1, 3).reshape(B, N, N_Q_HEADS * HEAD_DIM)
gqa_out = merged @ Wo
print(gqa_out.shape)
```
